# W8_02 · Use Bayesian optimization on five materials problems (advanced)

Apply the same BO loop to five datasets from real materials experiments. At each step, BO chooses an experiment from the dataset, then uses its recorded result to decide what to try next.

Data source: Liang et al., [*npj Computational Materials* **7**, 188 (2021)](https://doi.org/10.1038/s41524-021-00656-9).

In [ ]:
#@title 1. Setup — click ▶ to start
# Helper code for this lab. Nothing to edit here.

import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import norm
from sklearn.exceptions import ConvergenceWarning
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, Matern, WhiteKernel

# ---------- blanks (___) and ✅/❌ messages ----------
from IPython import get_ipython
from IPython.display import HTML, display


class BlankError(Exception):
    """A ___ is still in the cell."""


class _Blank:
    __array_ufunc__ = None  # numpy hands arithmetic back to us -> friendly stop

    def __repr__(self):
        return '___'

    def _stop(self, *args, **kwargs):
        raise BlankError('Replace every ___ in this cell with your answer, then run the cell again.')

    __add__ = __radd__ = __sub__ = __rsub__ = __mul__ = __rmul__ = __truediv__ = __rtruediv__ = _stop
    __pow__ = __rpow__ = __neg__ = __float__ = __int__ = __index__ = __bool__ = _stop
    __lt__ = __le__ = __gt__ = __ge__ = __call__ = __getitem__ = __iter__ = __len__ = _stop


___ = _Blank()


def has_blank(v):
    if isinstance(v, _Blank):
        return True
    if isinstance(v, dict):
        return any(has_blank(x) for x in v.values())
    if isinstance(v, (list, tuple)):
        return any(has_blank(x) for x in v)
    return False


def need(**values):
    """Stop with a friendly message if any value is (or contains) ___."""
    for name, v in values.items():
        if has_blank(v):
            raise BlankError(f'Fill in <b>{name}</b> (it still has ___), then run the cell again.')


def _say(color, text):
    display(HTML(f'<div style="border-left:5px solid {color};padding:6px 12px;margin:6px 0;font-size:14px">{text}</div>'))


def ok(text):
    _say('#3A9D5D', '✅ ' + text)


def bad(text):
    _say('#D9534F', '❌ ' + text)


def _friendly(shell, etype, evalue, tb, tb_offset=None):
    _say('#E8833A', '✋ ' + str(evalue))


if get_ipython() is not None:
    get_ipython().set_custom_exc((BlankError,), _friendly)

RAW = 'https://raw.githubusercontent.com/PV-Lab/Benchmarking/main/datasets/'
DOMAINS = {
    'P3HT': dict(file='P3HT_dataset.csv', goal='max', field='Polymer–nanotube composite film',
                 what='electrical conductivity (S/cm)', ref='Bash et al., Adv. Funct. Mater. 31, 2102606 (2021)'),
    'AgNP': dict(file='AgNP_dataset.csv', goal='min', field='Silver nanoparticle synthesis (microfluidic)',
                 what='spectrum mismatch "loss"', ref='Mekki-Berrada et al., npj Comput. Mater. 7, 55 (2021)'),
    'Perovskite': dict(file='Perovskite_dataset.csv', goal='min', field='Halide perovskite film (Cs/FA/MA)',
                       what='instability index (degradation)', ref='Sun et al., Matter 4, 1305 (2021)'),
    'CrossedBarrel': dict(file='Crossed barrel_dataset.csv', goal='max', field='3D-printed mechanical structure',
                          what='toughness', ref='Gongora et al., Sci. Adv. 6, eaaz1708 (2020)'),
    'AutoAM': dict(file='AutoAM_dataset.csv', goal='max', field='3D-printing process parameters',
                   what='print-quality score', ref='Deneault et al., MRS Bull. 46, 566 (2021)'),
}
BLUE, GREEN, GREY = '#2F6DB5', '#3A9D5D', '#8C929B'


def load(name):
    d = DOMAINS[name]
    df = pd.read_csv(d.get('path') or RAW + d['file'].replace(' ', '%20'), encoding='utf-8-sig')
    x, y = d.get('inputs') or list(df.columns[:-1]), d.get('target') or df.columns[-1]
    df = df.groupby(x, as_index=False)[y].mean()
    return df, x, y


def add_domain(name, path, inputs, target, goal='max', field='my data', what=None):
    """Use your own CSV (local path or URL): one row per experiment."""
    DOMAINS[name] = dict(path=path, inputs=inputs, target=target, goal=goal, field=field,
                         what=what or target, ref='your data')


def fit_gp(X, y, kernel=None):
    """kernel=None: learn the hyperparameters; otherwise reuse a fitted kernel (much faster)."""
    if kernel is None:
        kernel = (ConstantKernel(1.0, (1e-2, 1e2))
                  * Matern(length_scale=np.full(X.shape[1], 0.3), length_scale_bounds=(0.02, 10), nu=2.5)
                  + WhiteKernel(1e-3, (1e-6, 1e-1)))
        gp = GaussianProcessRegressor(kernel, normalize_y=True, n_restarts_optimizer=0, random_state=0)
    else:
        gp = GaussianProcessRegressor(kernel, normalize_y=True, optimizer=None)
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        gp.fit(X, y)
    return gp


def acquisition(mu, sd, best, acq='EI', beta=4.0, xi=0.01):
    if acq == 'UCB':
        return mu + np.sqrt(beta) * sd
    z = (mu - best - xi) / np.maximum(sd, 1e-12)
    return (mu - best - xi) * norm.cdf(z) + sd * norm.pdf(z)


def prepare(name):
    df, x, y = load(name)
    X = df[x].to_numpy(float)
    X = (X - X.min(0)) / np.where(np.ptp(X, 0) > 0, np.ptp(X, 0), 1)
    v = df[y].to_numpy(float) * (1 if DOMAINS[name]['goal'] == 'max' else -1)  # BO always maximises v
    return df, x, y, X, v


def campaign(X, v, budget, seed, acq='EI', n_init=5):
    """Order in which BO (or random, acq=None) picks rows of the pool."""
    rng = np.random.default_rng(seed)
    order = list(rng.permutation(len(X)))
    if acq is None:
        return order[:budget], None
    picked = order[:n_init]
    gp = None
    while len(picked) < min(budget, len(X)):
        refit = gp is None or len(picked) % 5 == 0  # relearn hyperparameters every 5 experiments
        gp = fit_gp(X[picked], v[picked], None if refit else gp.kernel_)
        rest = np.setdiff1d(np.arange(len(X)), picked)
        mu, sd = gp.predict(X[rest], return_std=True)
        s = (mu - v[picked].mean()) / v[picked].std(), sd / v[picked].std()
        a = acquisition(*s, (v[picked].max() - v[picked].mean()) / v[picked].std(), acq)
        picked.append(int(rest[np.argmax(a)]))
    return picked, gp


def curves(X, v, budget, seeds, acq):
    top = v >= np.quantile(v, 0.95)
    out = {}
    for label, a in [('Random', None), ('BO', acq)]:
        found, gps = [], []
        for s in range(seeds):
            picked, gp = campaign(X, v, budget, s, a)
            found.append(np.cumsum(top[picked]) / top.sum())
            gps.append(gp)
        out[label] = (np.array(found), gps)
    return out


def domains():
    """One-line description of each dataset and how its target values are spread."""
    rows = []
    fig, ax = plt.subplots(1, len(DOMAINS), figsize=(3.3 * len(DOMAINS), 2.9))
    for a, name in zip(np.atleast_1d(ax), DOMAINS):
        df, x, y, X, v = prepare(name)
        d = DOMAINS[name]
        rows.append({'dataset': name, 'field': d['field'], 'inputs': ', '.join(x), 'target': f"{d['goal']} {d['what']}",
                     'candidates': len(df), 'source': d['ref']})
        a.hist(df[y], bins=25, color=GREY)
        a.axvline(np.quantile(df[y], 0.95 if d['goal'] == 'max' else 0.05), color=BLUE, ls='--')
        a.set_title(f'{name} ({len(df)})', fontsize=10)
        a.set_xlabel(d['what'], fontsize=8)
        a.set_yticks([])
    fig.suptitle('Distribution of measured targets (dashed = top-5 % line)', y=1.04)
    plt.show()
    with pd.option_context('display.max_colwidth', 60):
        display(pd.DataFrame(rows).set_index('dataset'))


def check_name(name, acq='EI'):
    need(DATASET=name, ACQ=acq)
    if name not in DOMAINS:
        bad(f"DATASET must be one of {list(DOMAINS)} (with quotes).")
        return False
    if acq not in ('EI', 'UCB'):
        bad("ACQ must be 'EI' or 'UCB'.")
        return False
    return True


def run(name, acq='EI', budget=40, seeds=5):
    """BO vs random on one dataset + which inputs the GP found important."""
    if not check_name(name, acq):
        return
    df, x, y, X, v = prepare(name)
    budget = min(budget, len(df))
    res = curves(X, v, budget, seeds, acq)
    n = np.arange(1, budget + 1)
    fig, ax = plt.subplots(1, 2, figsize=(13, 4), gridspec_kw={'width_ratios': [1.4, 1]})
    for label, col in [('Random', GREEN), ('BO', BLUE)]:
        f = res[label][0]
        ax[0].plot(n, np.median(f, 0) * 100, color=col, lw=2, label=f'{label} ({acq})' if label == 'BO' else label)
        ax[0].fill_between(n, *np.percentile(f, [25, 75], 0) * 100, color=col, alpha=0.2)
    ax[0].set(xlabel='experiments', ylabel='top-5 % candidates found (%)', title=f'{name}: {DOMAINS[name]["field"]}')
    ax[0].legend()
    ls = np.mean([gp.kernel_.k1.k2.length_scale for gp in res['BO'][1]], 0)
    imp = 1 / ls / (1 / ls).sum()
    ax[1].barh(x, imp, color=BLUE)
    ax[1].invert_yaxis()
    ax[1].set(xlabel='relative importance (1 / GP length scale)', title='Which inputs matter (ARD)')
    fig.tight_layout()
    plt.show()
    for label in ['Random', 'BO']:
        print(f'{label:6s}: {np.median(res[label][0][:, -1]) * 100:5.1f} % of the top-5 % found after {budget} experiments')


def run_all(acq='EI', budget=40, seeds=5):
    """The same loop on every dataset: share of the top-5 % candidates found within the budget."""
    if not check_name('P3HT', acq):
        return
    names = list(DOMAINS)
    fig, ax = plt.subplots(1, len(names), figsize=(3.4 * len(names), 3.2), sharey=True)
    rows = []
    for a, name in zip(np.atleast_1d(ax), names):
        df, x, y, X, v = prepare(name)
        b = min(budget, len(df))
        res = curves(X, v, b, seeds, acq)
        n = np.arange(1, b + 1)
        final = {}
        for label, col in [('Random', GREEN), ('BO', BLUE)]:
            f = res[label][0]
            a.plot(n, np.median(f, 0) * 100, color=col, lw=2, label=label)
            final[label] = np.median(f[:, -1]) * 100
        a.set(title=name, xlabel='experiments')
        rows.append({'dataset': name, 'experiments': b, 'Random: top-5 % found (%)': round(final['Random']),
                     'BO: top-5 % found (%)': round(final['BO'])})
    np.atleast_1d(ax)[0].set_ylabel('top-5 % found (%)')
    np.atleast_1d(ax)[0].legend()
    fig.tight_layout()
    plt.show()
    display(pd.DataFrame(rows).set_index('dataset'))


# ---------- write the loop yourself ----------
class Picks(list):
    """Row numbers of the experiments run so far; append() refuses a blank."""

    def append(self, i):
        need(next_experiment=i)
        super().append(int(i))


def data_for(name):
    need(DATASET=name)
    df, x, y, X, v = prepare(name)
    print(f'{name}: {len(X)} candidates x {X.shape[1]} inputs ({", ".join(x)}); target: {DOMAINS[name]["goal"]} {y}')
    return X, v


def first_picks(X, n=5, seed=0):
    return Picks(int(i) for i in np.random.default_rng(seed).permutation(len(X))[:n])


def ei(mu, sd, best, xi=0.0):
    sd = np.maximum(sd, 1e-12)
    z = (mu - best - xi) / sd
    return (mu - best - xi) * norm.cdf(z) + sd * norm.pdf(z)


def reference_loop(X, v, steps, n=5, seed=0):
    picked = first_picks(X, n, seed)
    for _ in range(steps):
        gp = fit_gp(X[picked], v[picked])
        mu, sd = gp.predict(X, return_std=True)
        score = ei(mu, sd, best=v[picked].max())
        score[picked] = -np.inf
        picked.append(int(np.argmax(score)))
    return picked


def check_loop(picked, name, n=5, seed=0):
    need(picked=list(picked), DATASET=name)
    df, x, y, X, v = prepare(name)
    ref = reference_loop(X, v, len(picked) - n, n, seed)
    if list(picked) != list(ref):
        k = next(i for i, (a, b) in enumerate(zip(picked, ref)) if a != b) if len(picked) == len(ref) else min(len(picked), len(ref))
        return bad(f'Your loop differs from the reference at experiment {k + 1}. '
                   'Check: best = the highest v among picked rows; append the row with the highest score.')
    top = v >= np.quantile(v, 0.95)
    n_exp = np.arange(1, len(picked) + 1)
    rnd = np.median([np.cumsum(top[first_picks(X, len(picked), s)]) for s in range(20)], 0)
    fig, ax = plt.subplots(figsize=(6.5, 3.6))
    ax.plot(n_exp, np.cumsum(top[list(picked)]), color=BLUE, lw=2, label='your BO loop')
    ax.plot(n_exp, rnd, color=GREEN, lw=2, label='random (median of 20)')
    ax.set(xlabel='experiments', ylabel=f'top-5 % candidates found (of {top.sum()})', title=f'{name}: your loop vs random')
    ax.legend()
    plt.show()
    ok(f'Your loop works: same choices as the reference. It found {int(top[list(picked)].sum())} of the {top.sum()} '
       f'top-5 % candidates in {len(picked)} experiments (random: about {rnd[-1]:.0f}).')

plt.rcParams['figure.dpi'] = 100
print('Ready.')

## 2. Five materials problems

In [ ]:
domains()

## 3. Choose a materials problem

Pick one of the five problems and compare the two plots.

- **Left plot:** Of all the candidates in the top 5% of the dataset, what fraction has BO found so far? The curve shows the median across five different starts.
- **Right plot:** Which inputs does the GP seem most sensitive to? A shorter length scale means the model expects the result to change more quickly as that input changes.

In [ ]:
DATASET = ___   # 'P3HT', 'AgNP', 'Perovskite', 'CrossedBarrel' or 'AutoAM'
ACQ = ___       # 'EI' or 'UCB'
run(DATASET, acq=ACQ, budget=40, seeds=5)

## 4. Write the BO loop
Five lines, two blanks.

In [ ]:
X, v = data_for('P3HT')             # X: inputs scaled to 0-1 (one row per candidate); v: target, higher = better
picked = first_picks(X, n=5)         # row numbers of 5 random first experiments

for step in range(20):
    gp = fit_gp(X[picked], v[picked])            # 1. fit the surrogate to what we measured
    mu, sd = gp.predict(X, return_std=True)      # 2. predict every candidate
    score = ei(mu, sd, best=___)                 # 3. EI needs the best value measured so far (hint: v[picked])
    score[picked] = -np.inf                      #    never repeat an experiment
    nxt = int(np.argmax(score))                  # 4. the candidate with the highest EI
    picked.append(___)                           # 5. "run" that experiment

check_loop(picked, 'P3HT')

In [ ]:
#@title 💡 Answer — double-click to show (run it if you are stuck)
X, v = data_for('P3HT')             # X: inputs scaled to 0-1 (one row per candidate); v: target, higher = better
picked = first_picks(X, n=5)         # row numbers of 5 random first experiments

for step in range(20):
    gp = fit_gp(X[picked], v[picked])            # 1. fit the surrogate to what we measured
    mu, sd = gp.predict(X, return_std=True)      # 2. predict every candidate
    score = ei(mu, sd, best=v[picked].max())     # 3. EI needs the best value measured so far (hint: v[picked])
    score[picked] = -np.inf                      #    never repeat an experiment
    nxt = int(np.argmax(score))                  # 4. the candidate with the highest EI
    picked.append(nxt)                           # 5. "run" that experiment

check_loop(picked, 'P3HT')

## 5. Same code on all five (about 15 s)

In [ ]:
run_all(acq='EI', budget=40, seeds=5)

## 6. How does the BO loop change for other experiments?

In this lab, BO picked one experiment at a time to improve one target. Other setups call for different ways to choose the next experiment.

| What do you want to do? | Approach | Materials example |
|---|---|---|
| Choose several experiments before seeing any results | **Batch BO** selects a group of candidates using a rule such as q-UCB or q-EI. | Choose three electrolyte recipes for one ALBATROSS round (W8_01). |
| Balance competing goals | **Multi-objective BO** finds trade-offs where improving one goal means giving up ground on another (the Pareto front). | Balance high conductivity with low processing temperature in Pd thin films. [MacLeod et al. (2022)](https://doi.org/10.1038/s41467-022-28580-6) |
| Improve a target while meeting a requirement | **Constrained BO** considers both predicted performance and the likelihood of meeting the requirement. | Search for promising perovskites while accounting for phase stability. [Sun et al. (2021)](https://doi.org/10.1016/j.matt.2021.01.008) |
| Decide when an expensive measurement is worthwhile | **Multi-fidelity BO** combines cheaper estimates with more reliable, costly measurements. | Study conductivity in P3HT–CNT composites. [Bash et al. (2021)](https://doi.org/10.1002/adfm.202102606) |
| Choose ingredients as well as numerical settings | **Mixed-variable BO** handles categories, such as solvent type, alongside numbers, such as temperature. | Optimize reaction conditions. [Shields et al. (2021)](https://doi.org/10.1038/s41586-021-03213-y) |